In [7]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
with open(
    "/home/chenzihao/workspace/cc2cc_test5/jupyter-notebook/new_dataset/gmtkn-def2.json"
) as f:
    json_data = json.load(f)

dft_type_list = ["b3lyp_ene", "b3lyp-d3bj_ene"]
data = pd.read_csv(
    "/home/chenzihao/workspace/cc2cc_test5/validate_hkqai/ccdft_def2-TZVPD__gmtkn-def2.csv"
)

basis_args = "def2-TZVPD"
print(basis_args)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for dft_type in dft_type_list:
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data[dft_type].to_numpy() * 627.5094733748099

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "cc": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(f"Warning: {i_molecule_name} not found in data file")
                    continue

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data csv file")
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    atomic_energy_cc = i_reaction["reference"]
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ]
                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} with name {data_subset[name_subset]["name"][argsort_atomic_energy_dft[i]]} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

header = dft_type_list + ["Processed"]

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

for dft_type in dft_type_list:
    mean_absolute_deviation_list = []

    for name_set, subset_list_ in full_subset_dict.items():
        subset_dft = []
        wtmad_1_dft = []
        wtmad_2_dft = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"

            if len(data_subset[name_subset]["dft"]) == 0:
                df_summary_subset.loc[i_subset, dft_type] = 0
                df_summary_subset.loc[i_subset, "Processed"] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                df_summary_subset.loc[i_subset, dft_type] = np.mean(
                    data_subset[name_subset]["dft"]
                )
                df_summary_subset.loc[i_subset, "Processed"] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["dft"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["dft"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['dft'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                subset_dft = np.append(subset_dft, data_subset[name_subset]["dft"])
                wtmad_1_dft = np.append(
                    wtmad_1_dft,
                    wtmad_1 * np.mean(data_subset[name_subset]["dft"]),
                )
                wtmad_2_dft = np.append(
                    wtmad_2_dft,
                    data_subset[name_subset]["dft"]
                    / np.mean(data_subset[name_subset]["cc"]),
                )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["dft"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)

        mean_subset.loc[name_set, dft_type] = np.mean(subset_dft)
        wtmad_1_subset.loc[name_set, dft_type] = np.mean(wtmad_1_dft)
        wtmad_2_subset.loc[name_set, dft_type] = np.sum(wtmad_2_dft)
        mean_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {dft_type}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        wtmad_2_subset.loc[name_set, dft_type] = (
            mean_absolute_deviation * wtmad_2_subset.loc[name_set, dft_type]
        )

    wtmad_1_subset.loc["summary", "Processed"] = "--"
    wtmad_2_subset.loc["summary", "Processed"] = "--"
    wtmad_1_subset.loc["summary", dft_type] = 0
    wtmad_2_subset.loc["summary", dft_type] = 0
    for name_set in full_subset_dict.keys():
        wtmad_1_subset.loc["summary", dft_type] += wtmad_1_subset.loc[
            name_set, dft_type
        ]
        wtmad_2_subset.loc["summary", dft_type] += wtmad_2_subset.loc[
            name_set, dft_type
        ]

print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# # save summary to csv with date
# df_summary_subset_ele.to_csv(f"../validate_hkqai_done/df_summary_subset_ele_{date}.csv")
# df_summary_subset.to_csv(f"../validate_hkqai_done/summary_subset_{date}.csv")
# mean_subset.to_csv(f"../validate_hkqai_done/mean_subset_{date}.csv")
# wtmad_1_subset.to_csv(f"../validate_hkqai_done/wtmad_1_subset_{date}.csv")
# wtmad_2_subset.to_csv(f"../validate_hkqai_done/wtmad_2_subset_{date}.csv")
# # save summary to excel with date
# df_summary_subset_ele.to_excel(
#     f"../validate_hkqai_done/df_summary_subset_ele_{date}.xlsx"
# )
# df_summary_subset.to_excel(f"../validate_hkqai_done/summary_subset_{date}.xlsx")
# mean_subset.to_excel(f"../validate_hkqai_done/mean_subset_{date}.xlsx")
# wtmad_1_subset.to_excel(f"../validate_hkqai_done/wtmad_1_subset_{date}.xlsx")
# wtmad_2_subset.to_excel(f"../validate_hkqai_done/wtmad_2_subset_{date}.xlsx")

def2-TZVPD
Top 1 with name 130 DFT: 27.611040291833746 kcal/mol
Top 2 with name 41 DFT: 23.57685816803246 kcal/mol
Top 3 with name 59 DFT: 18.465193503116268 kcal/mol
Top 4 with name 46 DFT: 16.66766832751034 kcal/mol
Top 5 with name 107 DFT: 15.832422816137068 kcal/mol
Top 6 with name 128 DFT: 14.61009749176634 kcal/mol
Top 7 with name 138 DFT: 13.297855736038827 kcal/mol
Top 8 with name 69 DFT: 11.406077101526307 kcal/mol
Top 9 with name 110 DFT: 10.347673006909531 kcal/mol
Top 10 with name 116 DFT: 10.09249907268395 kcal/mol
Top 11 with name 104 DFT: 9.494733162737077 kcal/mol
Top 12 with name 121 DFT: 8.97286860715127 kcal/mol
Top 13 with name 126 DFT: 8.6950050329892 kcal/mol
Top 14 with name 42 DFT: 7.896587854456868 kcal/mol
Top 15 with name 137 DFT: 7.588451093484878 kcal/mol
Top 16 with name 133 DFT: 7.502828200650754 kcal/mol
Top 17 with name 52 DFT: 7.398278573773268 kcal/mol
Top 18 with name 92 DFT: 7.265498392404112 kcal/mol
Top 19 with name 25 DFT: 6.318844065279563 kcal/

,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,4.562869,3.707215,DONE
sub2,15.625329,7.261946,7 / 9
sub3,3.212841,3.249051,6 / 7
sub4,2.419527,0.8884,DONE
sub5,1.638705,0.315302,3 / 9


wtmad_1


,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,5.297991,3.891416,DONE
sub2,8.238651,5.301846,7 / 9
sub3,3.333248,3.284615,6 / 7
sub4,15.178475,3.094262,DONE
sub5,11.446032,2.513874,3 / 9
summary,43.494396,18.086013,--


wtmad_2


,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,2.529777,2.087674,DONE
sub2,3.93566,2.3377,7 / 9
sub3,1.694074,1.706235,6 / 7
sub4,8.487102,1.908139,DONE
sub5,3.494829,0.678168,3 / 9
summary,20.141442,8.717916,--


Summary of Subset
MAE


,b3lyp_ene,b3lyp-d3bj_ene,Processed
W4_11,3.94796,3.269621,DONE
G21EA,3.216316,3.210707,DONE
G21IP,3.755797,3.761985,DONE
DIPCS10,4.41082,4.392293,DONE
PA26,2.02061,2.458886,DONE
SIE4x4,17.332887,17.762184,DONE
ALKBDE10,3.937475,3.67966,DONE
YBDE18,8.306348,4.576729,DONE
AL2X6,8.92625,2.589699,DONE
HEAVYSB11,7.618308,3.026647,DONE
